<a href="https://colab.research.google.com/github/moyan-li/MSSP607/blob/main/Assignments/assignment1/assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from google.colab import userdata
import os

github_token = userdata.get('Neumann')
owner = 'moyan-li' # Replace with the GitHub repository owner
repository = 'MSSP607' # Replace with the GitHub repository name

clone_url = f'https://{github_token}@github.com/{owner}/{repository}.git'

# Clone the repository
!git clone {clone_url}

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import trim_mean, median_abs_deviation, skew

Mounted at /content/drive
Cloning into 'MSSP607'...
remote: Enumerating objects: 304, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 304 (delta 45), reused 20 (delta 15), pack-reused 221 (from 1)
Receiving objects: 100% (304/304), 12.66 MiB | 8.88 MiB/s, done.
Resolving deltas: 100% (154/154), done.


In [2]:
df = pd.read_excel("/content/drive/MyDrive/MSSP607/Assignments/Assignment1/data_academic_performance.xlsx")

#File path is wrong.  SHould be /content/MSSP607/Assignments/assignment1/data_academic_performance.xlsx

print(df.columns.tolist())

# Data types, missing values and basic statistics
df.info()
display(df.isna().sum().sort_values(ascending=False).head(20))
display(df.describe(include="all").T)
# Data quality inspection and cleaning

# Check the variable with the most missing values
missing_summary = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame(name="missing_count")
)

missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(df) * 100
)

display(missing_summary.head(20))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/MSSP607/Assignments/Assignment1/data_academic_performance.xlsx'

In [ ]:
#2.1 Numerical descriptive statistics
score_cols = [
    "MAT_S11", "CR_S11", "CC_S11", "BIO_S11", "ENG_S11",
    "QR_PRO", "CR_PRO", "CC_PRO", "ENG_PRO", "WC_PRO",
    "FEP_PRO", "G_SC"
]

numeric_summary = (
    df[score_cols]
      .describe(percentiles=[0.25, 0.50, 0.75])
      .T
      .rename(columns={
          "count": "n",
          "mean": "mean",
          "std": "std",
          "min": "min",
          "25%": "Q1",
          "50%": "median",
          "75%": "Q3",
          "max": "max"
      })
)

display(numeric_summary.round(2))
display(numeric_summary.loc[["G_SC"]].round(2))



In [ ]:
#2.2 Visualize score distributions
# Figure A1: Histogram and density curve of G_SC
plt.figure(figsize=(9, 5))

sns.histplot(
    data=df,
    x="G_SC",
    bins=30,
    kde=True,
    color="#4C78A8"
)

plt.axvline(
    df["G_SC"].mean(),
    color="red",
    linestyle="--",
    label=f"Mean = {df['G_SC'].mean():.2f}"
)

plt.axvline(
    df["G_SC"].median(),
    color="green",
    linestyle="--",
    label=f"Median = {df['G_SC'].median():.2f}"
)

plt.title("Distribution of Global Score (G_SC)")
plt.xlabel("Global Score")
plt.ylabel("Number of Students")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# The distribution of various achievements
score_long = df[score_cols].melt(
    var_name="Score_Type",
    value_name="Score"
)

g = sns.displot(
    data=score_long,
    x="Score",
    col="Score_Type",
    col_wrap=3,
    bins=30,
    kde=True,
    height=3,
    facet_kws={"sharex": False, "sharey": False}
)

g.set_titles("{col_name}")
g.set_axis_labels("Score", "Count")
plt.show()

In [ ]:
s11_cols = ["MAT_S11", "CR_S11", "CC_S11", "BIO_S11", "ENG_S11"]

plt.figure(figsize=(11, 5))

sns.boxplot(data=df[s11_cols], color="#72B7B2")

plt.title("Distribution of Saber 11 Scores")
plt.xlabel("Subject")
plt.ylabel("Score")
plt.tight_layout()
plt.show()

In [ ]:
#2.3 Outliers and robust statistics

g_sc = df["G_SC"].dropna()

q1 = g_sc.quantile(0.25)
q3 = g_sc.quantile(0.75)
iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

outliers = df[
    (df["G_SC"] < lower_fence) |
    (df["G_SC"] > upper_fence)
]

robust_summary = pd.DataFrame({
    "Statistic": [
        "Mean",
        "Median",
        "Standard deviation",
        "Q1",
        "Q3",
        "IQR",
        "Lower IQR fence",
        "Upper IQR fence",
        "Number of IQR outliers",
        "Percent of IQR outliers",
        "5% trimmed mean",
        "MAD",
        "Skewness"
    ],
    "Value": [
        g_sc.mean(),
        g_sc.median(),
        g_sc.std(),
        q1,
        q3,
        iqr,
        lower_fence,
        upper_fence,
        len(outliers),
        len(outliers) / len(g_sc) * 100,
        trim_mean(g_sc, 0.05),
        median_abs_deviation(g_sc, scale=1),
        skew(g_sc)
    ]
})

display(robust_summary.round(3))

print(f"The number of abnormal values for the IQR of G_SC: {len(outliers)}")
print(f"Proportion of outliers:{len(outliers) / len(g_sc) * 100:.2f}%")

In [ ]:
# Visualization

plt.figure(figsize=(9, 3))

sns.boxplot(x=df["G_SC"], color="#F58518")

plt.axvline(lower_fence, color="red", linestyle="--", label="Lower IQR fence")
plt.axvline(upper_fence, color="red", linestyle="--", label="Upper IQR fence")

plt.title("Outlier Check for Global Score (G_SC)")
plt.xlabel("Global Score")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#2.4 Categorical descriptive statistics
categorical_cols = df.select_dtypes(include="object").columns.tolist()

category_summary = pd.DataFrame({
    "Variable": categorical_cols,
    "Non_missing": [df[col].notna().sum() for col in categorical_cols],
    "Unique_categories": [df[col].nunique(dropna=True) for col in categorical_cols],
    "Most_common_category": [
        df[col].mode().iloc[0] if not df[col].mode().empty else np.nan
        for col in categorical_cols
    ],
    "Most_common_frequency": [
        df[col].value_counts().iloc[0] if not df[col].value_counts().empty else np.nan
        for col in categorical_cols
    ]
})

display(category_summary)

In [ ]:
def category_frequency_table(column):
    result = (
        df[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="Count")
    )

    result["Percent"] = result["Count"] / len(df) * 100

    return result


important_categories = [
    "GENDER",
    "STRATUM",
    "EDU_MOTHER",
    "EDU_FATHER",
    "INTERNET",
    "COMPUTER",
    "SCHOOL_NAT",
    "SCHOOL_TYPE"
]

for col in important_categories:
    print(f"\n===== {col} =====")
    display(category_frequency_table(col).round(2))

In [ ]:
#3.1.Does parental education influence outcomes, and if so, how much?
mother_clean = df.copy()

mother_clean["EDU_MOTHER"] = (
    mother_clean["EDU_MOTHER"]
    .astype(str)
    .str.strip()
)
mother_clean = mother_clean[
    ~mother_clean["EDU_MOTHER"].isin(["Not sure", "0"])
].copy()
mother_summary = (
    mother_clean.groupby("EDU_MOTHER")["G_SC"]
    .agg(n="count", mean="mean", median="median", std="std")
    .sort_values("mean", ascending=False)
)

display(mother_summary)
#box
plt.figure(figsize=(11, 5))
sns.boxplot(data=mother_clean, x="EDU_MOTHER", y="G_SC")
plt.xticks(rotation=40, ha="right")
plt.title("Overall university-assessment performance by maternal education")
plt.tight_layout()
plt.show()
#bar
plt.figure(figsize=(11, 5))

sns.barplot(
    data=mother_clean,
    x="EDU_MOTHER",
    y="G_SC",
    errorbar=("ci", 95)
)

plt.xticks(rotation=40, ha="right")
plt.title("Mean global score by maternal education level")
plt.xlabel("Maternal education level")
plt.ylabel("Mean global score (G_SC)")
plt.tight_layout()
plt.show()

In [ ]:
#anova: The larger the F value is, the stronger the difference between the groups will be.
groups = [
    group["G_SC"].dropna()
    for _, group in mother_clean.groupby("EDU_MOTHER")
]

f_stat, p_value = stats.f_oneway(*groups)
print(f"ANOVA: F = {f_stat:.2f}, p = {p_value:.4g}")

#anova:η²（eta- squared）
valid = mother_clean[["EDU_MOTHER", "G_SC"]].dropna()

grand_mean = valid["G_SC"].mean()
ss_between = sum(
    len(g) * (g["G_SC"].mean() - grand_mean) ** 2
    for _, g in valid.groupby("EDU_MOTHER")
)
ss_total = ((valid["G_SC"] - grand_mean) ** 2).sum()
eta_sq = ss_between / ss_total

print(f"Eta squared = {eta_sq:.3f}")

In [ ]:
#father
father_clean = df[df["EDU_FATHER"] != "Not sure"].copy()
father_clean = father_clean[
    ~father_clean["EDU_FATHER"].isin(["Not sure", "0"])
].copy()
father_summary = (
    df.groupby("EDU_FATHER" )["G_SC"]
      .agg(n="count", mean="mean", median="median", std="std")
      .sort_values("mean", ascending=False)
)
display(father_summary)
#box
plt.figure(figsize=(11, 5))
sns.boxplot(data=father_clean, x="EDU_FATHER", y="G_SC")
plt.xticks(rotation=40, ha="right")
plt.title("Overall university-assessment performance by Faternal education")
plt.tight_layout()
plt.show()
#bar
plt.figure(figsize=(11, 5))

sns.barplot(
    data=mother_clean,
    x="EDU_FATHER",
    y="G_SC",
    errorbar=("ci", 95)
)

plt.xticks(rotation=40, ha="right")
plt.title("Mean global score by Faternal education level")
plt.xlabel("Faternal education level")
plt.ylabel("Mean global score (G_SC)")
plt.tight_layout()
plt.show()
#anova
father_groups = [
    group["G_SC"].dropna()
    for _, group in father_clean.groupby("EDU_FATHER")
]

f_stat, p_value = stats.f_oneway(*father_groups)
print(f"ANOVA: F = {f_stat:.2f}, p = {p_value:.4g}")

#anova: η²（eta- squared）
valid = father_clean[["EDU_FATHER", "G_SC"]].dropna()

grand_mean = valid["G_SC"].mean()
ss_between = sum(
    len(g) * (g["G_SC"].mean() - grand_mean) ** 2
    for _, g in valid.groupby("EDU_FATHER")
)
ss_total = ((valid["G_SC"] - grand_mean) ** 2).sum()
eta_sq = ss_between / ss_total

print(f"Eta squared = {eta_sq:.3f}")


In [ ]:
#3.2.Which groups/stratum performs best? Are there differences between groups/stratums? Explain your analysis.
stratum_summary = (
    df.groupby("STRATUM")["G_SC"]
      .agg(n="count", mean="mean", median="median", std="std")
      .sort_values("mean", ascending=False)
)

display(stratum_summary)

plt.figure(figsize=(9, 5))
sns.barplot(
    data=df,
    x="STRATUM",
    y="G_SC",
    errorbar=("ci", 95)
)
plt.title("Mean overall performance by socioeconomic stratum")
plt.ylabel("Global Score")
plt.show()

stratum_groups = [
    group["G_SC"].dropna()
    for _, group in df.groupby("STRATUM")
]
f_stat, p_value = stats.f_oneway(*stratum_groups)
print(f"ANOVA: F = {f_stat:.2f}, p = {p_value:.4g}")

In [ ]:
#The F-test was significant, indicating that there are differences between the groups.
#Examine which group performed the best.
tmp = df[["STRATUM", "G_SC"]].dropna()

tmp["STRATUM"] = tmp["STRATUM"].astype(str)
tmp["G_SC"] = pd.to_numeric(tmp["G_SC"], errors="coerce")

tukey = pairwise_tukeyhsd(
    endog=tmp["G_SC"],
    groups=tmp["STRATUM"],
    alpha=0.05
)

print(tukey)

In [ ]:
#3.3.How does each gender perform in the global assessment?
gender_summary = (
    df.groupby("GENDER")["G_SC"]
      .agg(n="count", mean="mean", median="median", std="std")
      .sort_values("mean", ascending=False)
)

display(gender_summary)

plt.figure(figsize=(6, 5))
sns.barplot(
    data=df,
    x="GENDER",
    y="G_SC",
    errorbar=("ci", 95)
)
plt.title("Overall performance by gender")
plt.show()

gender_groups = [
    group["G_SC"].dropna()
    for _, group in df.groupby("GENDER")
]

#Welch t-test for two groups
female_scores = df.loc[df["GENDER"] == "F", "G_SC"].dropna()
male_scores = df.loc[df["GENDER"] == "M", "G_SC"].dropna()

t_stat, p_value = stats.ttest_ind(
    female_scores,
    male_scores,
    equal_var=False
)

print(f"Welch t-test: t = {t_stat:.2f}, p = {p_value:.4g}")

#cohen's d
#|d| ≈ 0.20：Minor effect; |d| ≈ 0.50：Medium effect; |d| ≈ 0.80: Strong effect。
g1, g2 = gender_groups

pooled_sd = np.sqrt(
    ((len(g1) - 1) * g1.var(ddof=1) + (len(g2) - 1) * g2.var(ddof=1))
    / (len(g1) + len(g2) - 2)
)

cohens_d = (g1.mean() - g2.mean()) / pooled_sd
print(f"Cohen's d = {cohens_d:.3f}")

#Summary:
#Multiple groups:
#1. Group mean table: Answer "Which stratum performed the best";
#2. ANOVA F-test: Answer "Is there at least an inter-group difference overall?" p < 0.05 is significant
#In the ANOVA test, η² (eta-squared): η² ≈ 0.01: small effect; η² ≈ 0.06: medium effect; η² ≈ 0.14: large effect
#F / p value answers "Does the difference exist?"; η² answers "How large is the difference"
#3. When F is significant, Tukey HSD: Answer "Which two strata have a significant difference specifically"
#Two groups:
#1. Group mean table: Answer "Which gender performed better";
#2. t-test (to check the inter-group difference) p < 0.05 is significant
#3. Calculate Cohen's absolute d value: |d| ≈ 0.20: small effect; |d| ≈ 0.50: medium effect; |d| ≈ 0.80: large effect
#Explanation of the meaning of effect size Cohen's d: Avoid interpreting very small score differences too importantly due to a large sample size.


In [ ]:
#3.4.Use your analysis to understand the results.  What did you find interesting, and can you make any recommendations or inferences?
program_counts = df["ACADEMIC_PROGRAM"].value_counts()
display(program_counts.to_frame(name="n"))

#A professional with a sample size of no less than 30 people retained
valid_programs = program_counts[program_counts >= 30].index

program_df = df[
    df["ACADEMIC_PROGRAM"].isin(valid_programs)
].copy()

print("numbers of students：", len(program_df))
print("numbers of programs：", program_df["ACADEMIC_PROGRAM"].nunique())

#Descriptive statistics table for each engineering discipline
program_summary = (
    program_df.groupby("ACADEMIC_PROGRAM")["G_SC"]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std"
    )
    .sort_values("mean", ascending=False)
)

display(program_summary.round(2))

# ANOVA: F test
program_groups = [
    group["G_SC"].dropna()
    for _, group in program_df.groupby("ACADEMIC_PROGRAM")
]

f_stat, p_value = stats.f_oneway(*program_groups)

df_between = len(program_groups) - 1
df_within = len(program_df) - len(program_groups)

print(
    f"One-way ANOVA: "
    f"F({df_between}, {df_within}) = {f_stat:.2f}, "
    f"p = {p_value:.4g}"
)

#eta-squared（η²）

grand_mean = program_df["G_SC"].mean()

ss_between = sum(
    len(group) * (group["G_SC"].mean() - grand_mean) ** 2
    for _, group in program_df.groupby("ACADEMIC_PROGRAM")
)

ss_total = (
    (program_df["G_SC"] - grand_mean) ** 2
).sum()

eta_sq_program = ss_between / ss_total

print(f"Eta-squared (η²) = {eta_sq_program:.3f}")

# mean bar
program_order = program_summary.index.tolist()

plt.figure(figsize=(14, 6))

sns.barplot(
    data=program_df,
    x="ACADEMIC_PROGRAM",
    y="G_SC",
    order=program_order,
    errorbar=("ci", 95),
    color="#4C78A8"
)

plt.axhline(
    df["G_SC"].mean(),
    color="red",
    linestyle="--",
    label=f"Overall mean = {df['G_SC'].mean():.2f}"
)

plt.title("Mean Global Score by Academic Program")
plt.xlabel("Academic Program")
plt.ylabel("Mean Global Score (G_SC)")
plt.xticks(rotation=60, ha="right")
plt.legend()
plt.tight_layout()
plt.show()